In [48]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA

In [49]:
def load_and_preprocess_data(filepath="/content/first_25000_rows.csv"):

    # Load data
    df = pd.read_csv(filepath)

    # Convert timestamp to datetime
    df['timestamp'] = pd.to_datetime(df['ts_event'])

    # Sort by symbol and timestamp to ensure chronological order
    df = df.sort_values(['symbol', 'timestamp'])

    return df


In [50]:
def compute_order_flows(df, M=10):
    """
    Compute bid and ask order flows for each level of the order book.

    Parameters:
    - df (pd.DataFrame): DataFrame with order book data for a single symbol.
    - M (int): Number of order book levels to process (default is 10).

    Returns:
    - pd.DataFrame: DataFrame with added order flow columns.
    """
    for m in range(1, M + 1):
        level = m - 1
        bid_px_col = f'bid_px_{level:02d}'
        bid_sz_col = f'bid_sz_{level:02d}'
        ask_px_col = f'ask_px_{level:02d}'
        ask_sz_col = f'ask_sz_{level:02d}'

        price_change_b = df[bid_px_col] - df[bid_px_col].shift(1)
        price_change_a = df[ask_px_col] - df[ask_px_col].shift(1)

        df[f'OF_{m}_b'] = np.where(price_change_b > 0, df[bid_sz_col],
                                   np.where(price_change_b == 0, df[bid_sz_col] - df[bid_sz_col].shift(1),
                                            -df[bid_sz_col].shift(1)))

        df[f'OF_{m}_a'] = np.where(price_change_a < 0, df[ask_sz_col],
                                   np.where(price_change_a == 0, df[ask_sz_col] - df[ask_sz_col].shift(1),
                                            -df[ask_sz_col].shift(1)))


        df[f'OF_{m}_b'] = df[f'OF_{m}_b'].fillna(0)
        df[f'OF_{m}_a'] = df[f'OF_{m}_a'].fillna(0)

        df[f'delta_{m}'] = df[f'OF_{m}_b'] - df[f'OF_{m}_a']

    return df

In [51]:
def aggregate_to_time_buckets(df, M=10, freq='1min'):
    """
    Aggregate order flows into time buckets.

    Parameters:
    - df (pd.DataFrame): DataFrame with order flows.
    - M (int): Number of order book levels (default is 10).
    - freq (str): Frequency for time bucketing (default is '1min').

    Returns:
    - pd.DataFrame: DataFrame with aggregated OFI per time bucket.
    """
    df['time_bucket'] = df['timestamp'].dt.floor(freq)
    ofi_df = df.groupby(['symbol', 'time_bucket']).agg({f'delta_{m}': 'sum' for m in range(1, M + 1)}).reset_index()
    ofi_df.columns = ['symbol', 'time_bucket'] + [f'ofi_{m}' for m in range(1, M + 1)]
    return ofi_df

In [52]:
def compute_ofi_features(ofi_df, M=10):
    """
    Compute Best-Level, Multi-Level, Integrated, and Cross-Asset OFI features.

    Parameters:
    - ofi_df (pd.DataFrame): DataFrame with aggregated OFI data.
    - M (int): Number of order book levels (default is 10).

    Returns:
    - pd.DataFrame: DataFrame with computed OFI features.
    """
    features_list = []
    for symbol, group in ofi_df.groupby('symbol'):
        ofi_matrix = group[[f'ofi_{m}' for m in range(1, M + 1)]].values
        pca = PCA(n_components=1)
        integrated_ofi = pca.fit_transform(ofi_matrix).flatten()
        best_level_ofi = group['ofi_1'].values
        multi_level_ofi = ofi_matrix.sum(axis=1)
        group_df = pd.DataFrame({
            'time_bucket': group['time_bucket'],
            'symbol': symbol,
            'best_level_ofi': best_level_ofi,
            'multi_level_ofi': multi_level_ofi,
            'integrated_ofi': integrated_ofi
        })
        features_list.append(group_df)

    features_df = pd.concat(features_list)

    # Corrección: verificar si hay más de un símbolo para Cross-Asset OFI
    cross_asset_data = features_df.pivot(index='time_bucket', columns='symbol', values='best_level_ofi')
    mean_ofi = cross_asset_data.mean(axis=1)
    N = cross_asset_data.shape[1]
    if N == 1:
        print("Only one asset detected. Cross-asset OFI cannot be calculated.")
        cross_asset_ofi = np.full_like(cross_asset_data.values, np.nan)
    else:
        cross_asset_ofi = (N * mean_ofi.values[:, None] - cross_asset_data.values) / (N - 1)

    cross_asset_ofi_df = pd.DataFrame(cross_asset_ofi, index=cross_asset_data.index, columns=cross_asset_data.columns)
    cross_asset_ofi_df = cross_asset_ofi_df.reset_index().melt(id_vars='time_bucket',
                                                              var_name='symbol',
                                                              value_name='cross_asset_ofi')

    features_df = features_df.merge(cross_asset_ofi_df, on=['time_bucket', 'symbol'], how='left')
    return features_df

In [53]:
def save_output(df, output_path):
    """
    Save the computed features to a CSV file.

    Parameters:
    - df (pd.DataFrame): DataFrame with computed features.
    - output_path (str): Path to save the output file.

    Returns:
    - pd.DataFrame: The final DataFrame for reference.
    """
    output_df = df[['time_bucket', 'symbol', 'best_level_ofi', 'multi_level_ofi', 'integrated_ofi', 'cross_asset_ofi']]
    output_df.to_csv(output_path, index=False)
    return output_df

In [54]:
def main(file_path='first_25000_rows.csv', output_path='ofi_features.csv', M=10, freq='1min'):
    """
    Main function to compute OFI features and save the results.

    Parameters:
    - file_path (str): Path to the input dataset.
    - output_path (str): Path to save the output file.
    - M (int): Number of order book levels (default is 10).
    - freq (str): Frequency for time bucketing (default is '1min').

    Returns:
    - pd.DataFrame: DataFrame with computed features.
    """
    df = load_and_preprocess_data(file_path)
    df = df.groupby('symbol')[df.columns].apply(lambda g: compute_order_flows(g)).reset_index(drop=True)
    ofi_df = aggregate_to_time_buckets(df, M=M, freq=freq)
    features_df = compute_ofi_features(ofi_df, M=M)
    output_df = save_output(features_df, output_path)
    return output_df

if __name__ == "__main__":
    output_df = main()
    print(output_df.head())

Only one asset detected. Cross-asset OFI cannot be calculated.
                time_bucket symbol  best_level_ofi  multi_level_ofi  \
0 2024-10-21 11:54:00+00:00   AAPL          -195.0           1438.0   
1 2024-10-21 11:55:00+00:00   AAPL          -916.0           1089.0   
2 2024-10-21 11:56:00+00:00   AAPL           199.0           1396.0   
3 2024-10-21 11:57:00+00:00   AAPL           201.0           -632.0   
4 2024-10-21 11:58:00+00:00   AAPL           863.0           3550.0   

   integrated_ofi  cross_asset_ofi  
0      439.351964              NaN  
1      801.902115              NaN  
2      615.033833              NaN  
3       97.170772              NaN  
4     1317.312885              NaN  


In [55]:
print(output_df)

                 time_bucket symbol  best_level_ofi  multi_level_ofi  \
0  2024-10-21 11:54:00+00:00   AAPL          -195.0           1438.0   
1  2024-10-21 11:55:00+00:00   AAPL          -916.0           1089.0   
2  2024-10-21 11:56:00+00:00   AAPL           199.0           1396.0   
3  2024-10-21 11:57:00+00:00   AAPL           201.0           -632.0   
4  2024-10-21 11:58:00+00:00   AAPL           863.0           3550.0   
..                       ...    ...             ...              ...   
66 2024-10-21 13:00:00+00:00   AAPL          -989.0          -1887.0   
67 2024-10-21 13:01:00+00:00   AAPL         -1122.0          -6024.0   
68 2024-10-21 13:02:00+00:00   AAPL         -4344.0         -24516.0   
69 2024-10-21 13:03:00+00:00   AAPL           181.0           5674.0   
70 2024-10-21 13:04:00+00:00   AAPL          -219.0          -5073.0   

    integrated_ofi  cross_asset_ofi  
0       439.351964              NaN  
1       801.902115              NaN  
2       615.033833   